In [1]:
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [2]:
# Load saved artifacts

model = joblib.load(
    "../models/xgboost_candidate.pkl"
)

scaler = joblib.load(
    "../models/scaler.pkl"
)

split_data = joblib.load(
    "../models/train_test_split.pkl"
)

X_test = split_data["X_test"]
y_test = split_data["y_test"]

print("Model loaded:", type(model).__name__)
print("Test samples:", len(X_test))
print("Fraud cases:", int(y_test.sum()))
print("Legitimate cases:", int((y_test == 0).sum()))

Model loaded: XGBClassifier
Test samples: 56746
Fraud cases: 95
Legitimate cases: 56651


In [3]:
MODEL_FEATURES = [
    "Time",
    "V1", "V2", "V3", "V4", "V5",
    "V6", "V7", "V8", "V9", "V10",
    "V11", "V12", "V13", "V14", "V15",
    "V16", "V17", "V18", "V19", "V20",
    "V21", "V22", "V23", "V24", "V25",
    "V26", "V27", "V28",
    "Amount"
]

X_test_model = X_test[MODEL_FEATURES]

X_test_scaled = scaler.transform(
    X_test_model
)

print("Test data shape:", X_test_scaled.shape)

Test data shape: (56746, 30)


In [4]:
# Generate fraud probabilities

y_probability = model.predict_proba(
    X_test_scaled
)[:, 1]

# Standard binary prediction at 0.5 threshold

y_pred = (
    y_probability >= 0.5
).astype(int)

print("Predictions generated successfully.")

print("\nProbability range:")
print(
    "Minimum:", y_probability.min()
)

print(
    "Maximum:", y_probability.max()
)

print(
    "Predicted fraud:", y_pred.sum()
)

Predictions generated successfully.

Probability range:
Minimum: 1.9480376e-07
Maximum: 0.99902904
Predicted fraud: 73


In [5]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


# =========================================================
# FINAL HELD-OUT TEST METRICS
# =========================================================

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

pr_auc = average_precision_score(
    y_test,
    y_probability
)


# =========================================================
# CONFUSION MATRIX
# =========================================================

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred
).ravel()


# =========================================================
# DISPLAY RESULTS
# =========================================================

print("=" * 60)
print("FINAL HELD-OUT TEST SET EVALUATION")
print("=" * 60)

print(f"Test Samples : {len(y_test)}")
print(f"Actual Fraud : {int(y_test.sum())}")
print()

print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc_auc:.4f}")
print(f"PR-AUC       : {pr_auc:.4f}")

print()

print("Confusion Matrix")
print("----------------")
print(f"TN : {tn}")
print(f"FP : {fp}")
print(f"FN : {fn}")
print(f"TP : {tp}")

FINAL HELD-OUT TEST SET EVALUATION
Test Samples : 56746
Actual Fraud : 95

Precision    : 0.9726
Recall       : 0.7474
F1 Score     : 0.8452
ROC-AUC      : 0.9775
PR-AUC       : 0.8292

Confusion Matrix
----------------
TN : 56649
FP : 2
FN : 24
TP : 71


In [8]:
# =========================================================
# COST-SENSITIVE RISK POLICY EVALUATION
# =========================================================

THRESHOLD_MEDIUM = 0.05
THRESHOLD_HIGH = 0.20

FN_COST = 10
FP_COST = 1


# ---------------------------------------------------------
# Risk levels
# ---------------------------------------------------------

risk_level = np.select(
    [
        y_probability < THRESHOLD_MEDIUM,

        (
            (y_probability >= THRESHOLD_MEDIUM) &
            (y_probability < THRESHOLD_HIGH)
        ),

        y_probability >= THRESHOLD_HIGH
    ],
    [
        "LOW",
        "MEDIUM",
        "HIGH"
    ],
    default="UNKNOWN"
)


# ---------------------------------------------------------
# Actions
# ---------------------------------------------------------

action = np.select(
    [
        y_probability < THRESHOLD_MEDIUM,

        (
            (y_probability >= THRESHOLD_MEDIUM) &
            (y_probability < THRESHOLD_HIGH)
        ),

        y_probability >= THRESHOLD_HIGH
    ],
    [
        "APPROVE",
        "REVIEW",
        "BLOCK"
    ],
    default="UNKNOWN"
)


# ---------------------------------------------------------
# Create policy DataFrame
# ---------------------------------------------------------

policy_df = pd.DataFrame({

    "actual": y_test.values,

    "probability": y_probability,

    "risk_level": risk_level,

    "action": action

})


# =========================================================
# TRANSACTION ROUTING
# =========================================================

print("=" * 60)
print("COST-SENSITIVE RISK POLICY EVALUATION")
print("=" * 60)


print("\nTransaction Routing")
print("--------------------")

routing = (
    policy_df["action"]
    .value_counts()
    .reindex(
        ["APPROVE", "REVIEW", "BLOCK"],
        fill_value=0
    )
)

print(routing)


# =========================================================
# ACTUAL FRAUD BY ACTION
# =========================================================

print("\nActual Fraud Cases by Action")
print("-----------------------------")

fraud_by_action = (
    policy_df[
        policy_df["actual"] == 1
    ]["action"]
    .value_counts()
    .reindex(
        ["APPROVE", "REVIEW", "BLOCK"],
        fill_value=0
    )
)

print(fraud_by_action)


# =========================================================
# LEGITIMATE TRANSACTIONS BY ACTION
# =========================================================

print("\nLegitimate Cases by Action")
print("---------------------------")

legitimate_by_action = (
    policy_df[
        policy_df["actual"] == 0
    ]["action"]
    .value_counts()
    .reindex(
        ["APPROVE", "REVIEW", "BLOCK"],
        fill_value=0
    )
)

print(legitimate_by_action)


# =========================================================
# FRAUD APPROVED
# =========================================================

fraud_approved = len(
    policy_df[
        (policy_df["actual"] == 1) &
        (policy_df["action"] == "APPROVE")
    ]
)


# =========================================================
# FRAUD SENT TO REVIEW
# =========================================================

fraud_review = len(
    policy_df[
        (policy_df["actual"] == 1) &
        (policy_df["action"] == "REVIEW")
    ]
)


# =========================================================
# FRAUD BLOCKED
# =========================================================

fraud_blocked = len(
    policy_df[
        (policy_df["actual"] == 1) &
        (policy_df["action"] == "BLOCK")
    ]
)


# =========================================================
# LEGITIMATE TRANSACTIONS BLOCKED
# =========================================================

legitimate_blocked = len(
    policy_df[
        (policy_df["actual"] == 0) &
        (policy_df["action"] == "BLOCK")
    ]
)


# =========================================================
# COST CALCULATION
# =========================================================

total_cost = (
    legitimate_blocked * FP_COST
    +
    fraud_approved * FN_COST
)


# =========================================================
# FINAL RESULTS
# =========================================================

print("\nPolicy Outcomes")
print("----------------")

print(
    f"Fraud approved       : {fraud_approved}"
)

print(
    f"Fraud review         : {fraud_review}"
)

print(
    f"Fraud blocked        : {fraud_blocked}"
)

print(
    f"Legitimate blocked   : {legitimate_blocked}"
)

print(
    f"\nFalse Negative Cost  : {fraud_approved} × {FN_COST}"
)

print(
    f"False Positive Cost  : {legitimate_blocked} × {FP_COST}"
)

print(
    f"\nTotal Prototype Cost : {total_cost}"
)

print(
    f"FN:FP Cost Ratio     : {FN_COST}:1"
)

COST-SENSITIVE RISK POLICY EVALUATION

Transaction Routing
--------------------
action
APPROVE    56653
REVIEW        18
BLOCK         75
Name: count, dtype: int64

Actual Fraud Cases by Action
-----------------------------
action
APPROVE    18
REVIEW      4
BLOCK      73
Name: count, dtype: int64

Legitimate Cases by Action
---------------------------
action
APPROVE    56635
REVIEW        14
BLOCK          2
Name: count, dtype: int64

Policy Outcomes
----------------
Fraud approved       : 18
Fraud review         : 4
Fraud blocked        : 73
Legitimate blocked   : 2

False Negative Cost  : 18 × 10
False Positive Cost  : 2 × 1

Total Prototype Cost : 182
FN:FP Cost Ratio     : 10:1


In [9]:
# =========================================================
# FRAUD LEAKAGE ANALYSIS
# =========================================================

fraud_approved_df = policy_df[
    (policy_df["actual"] == 1) &
    (policy_df["action"] == "APPROVE")
].copy()

print("=" * 60)
print("FRAUD LEAKAGE ANALYSIS")
print("=" * 60)

print(
    "Fraud transactions approved:",
    len(fraud_approved_df)
)

print("\nFraud probability statistics:")

print(
    fraud_approved_df["probability"].describe()
)

print("\nLowest fraud probabilities among missed fraud:")

print(
    fraud_approved_df[
        ["probability"]
    ]
    .sort_values("probability")
    .head(10)
)

FRAUD LEAKAGE ANALYSIS
Fraud transactions approved: 18

Fraud probability statistics:
count    18.000000
mean      0.004163
std       0.009780
min       0.000006
25%       0.000038
50%       0.000087
75%       0.003562
max       0.040543
Name: probability, dtype: float64

Lowest fraud probabilities among missed fraud:
       probability
5187      0.000006
47606     0.000009
29007     0.000017
56082     0.000033
42473     0.000037
22495     0.000039
42747     0.000058
54283     0.000060
27361     0.000068
4480      0.000105


In [10]:
# =========================================================
# THRESHOLD COST ANALYSIS
# =========================================================

FN_COST = 10
FP_COST = 1


thresholds = np.arange(
    0.01,
    0.51,
    0.01
)


results = []


for threshold in thresholds:

    predictions = (
        y_probability >= threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()


    precision_value = precision_score(
        y_test,
        predictions,
        zero_division=0
    )


    recall_value = recall_score(
        y_test,
        predictions,
        zero_division=0
    )


    f1_value = f1_score(
        y_test,
        predictions,
        zero_division=0
    )


    total_cost = (
        fp * FP_COST
        +
        fn * FN_COST
    )


    results.append({

        "threshold": threshold,

        "precision": precision_value,

        "recall": recall_value,

        "f1": f1_value,

        "TN": tn,

        "FP": fp,

        "FN": fn,

        "TP": tp,

        "cost": total_cost

    })


threshold_results = pd.DataFrame(
    results
)


# =========================================================
# BEST THRESHOLD BY COST
# =========================================================

best_cost_row = (
    threshold_results
    .sort_values("cost")
    .iloc[0]
)


print("=" * 60)
print("THRESHOLD OPTIMIZATION")
print("=" * 60)

print(
    f"Best threshold by cost: "
    f"{best_cost_row['threshold']:.2f}"
)

print(
    f"Cost: "
    f"{best_cost_row['cost']:.0f}"
)

print(
    f"Precision: "
    f"{best_cost_row['precision']:.4f}"
)

print(
    f"Recall: "
    f"{best_cost_row['recall']:.4f}"
)

print(
    f"F1: "
    f"{best_cost_row['f1']:.4f}"
)

print()

print("Confusion Matrix")
print("----------------")

print(
    f"TN: {best_cost_row['TN']:.0f}"
)

print(
    f"FP: {best_cost_row['FP']:.0f}"
)

print(
    f"FN: {best_cost_row['FN']:.0f}"
)

print(
    f"TP: {best_cost_row['TP']:.0f}"
)

THRESHOLD OPTIMIZATION
Best threshold by cost: 0.04
Cost: 187
Precision: 0.8211
Recall: 0.8211
F1: 0.8211

Confusion Matrix
----------------
TN: 56634
FP: 17
FN: 17
TP: 78


In [11]:
# =========================================================
# TWO-THRESHOLD RISK POLICY OPTIMIZATION
# =========================================================

FN_COST = 10
FP_COST = 1

threshold_values = np.arange(
    0.01,
    0.51,
    0.01
)

policy_results = []


for medium_threshold in threshold_values:

    for high_threshold in threshold_values:

        # High threshold must be above medium threshold
        if high_threshold <= medium_threshold:
            continue


        # -------------------------------------------------
        # Assign risk levels
        # -------------------------------------------------

        risk = np.where(
            y_probability < medium_threshold,
            "APPROVE",
            np.where(
                y_probability < high_threshold,
                "REVIEW",
                "BLOCK"
            )
        )


        # -------------------------------------------------
        # Actual fraud
        # -------------------------------------------------

        fraud_approved = np.sum(
            (y_test.values == 1) &
            (risk == "APPROVE")
        )


        fraud_review = np.sum(
            (y_test.values == 1) &
            (risk == "REVIEW")
        )


        fraud_blocked = np.sum(
            (y_test.values == 1) &
            (risk == "BLOCK")
        )


        # -------------------------------------------------
        # Legitimate transactions blocked
        # -------------------------------------------------

        legitimate_blocked = np.sum(
            (y_test.values == 0) &
            (risk == "BLOCK")
        )


        # -------------------------------------------------
        # Prototype cost
        #
        # Fraud approved = FN
        # Legitimate blocked = FP
        #
        # REVIEW is manually investigated and therefore
        # is not counted as FN or FP in this prototype cost.
        # -------------------------------------------------

        total_cost = (
            fraud_approved * FN_COST
            +
            legitimate_blocked * FP_COST
        )


        # -------------------------------------------------
        # Fraud capture
        # -------------------------------------------------

        fraud_capture_rate = (
            fraud_blocked + fraud_review
        ) / y_test.sum()


        policy_results.append({

            "medium_threshold":
                medium_threshold,

            "high_threshold":
                high_threshold,

            "cost":
                total_cost,

            "fraud_approved":
                fraud_approved,

            "fraud_review":
                fraud_review,

            "fraud_blocked":
                fraud_blocked,

            "legitimate_blocked":
                legitimate_blocked,

            "fraud_capture_rate":
                fraud_capture_rate

        })


policy_results_df = pd.DataFrame(
    policy_results
)


# =========================================================
# BEST POLICY
# =========================================================

best_policy = (
    policy_results_df
    .sort_values(
        [
            "cost",
            "fraud_capture_rate"
        ],
        ascending=[
            True,
            False
        ]
    )
    .iloc[0]
)


print("=" * 60)
print("TWO-THRESHOLD POLICY OPTIMIZATION")
print("=" * 60)

print(
    f"Best medium threshold : "
    f"{best_policy['medium_threshold']:.2f}"
)

print(
    f"Best high threshold   : "
    f"{best_policy['high_threshold']:.2f}"
)

print(
    f"Prototype cost        : "
    f"{best_policy['cost']:.0f}"
)

print()

print(
    f"Fraud approved        : "
    f"{best_policy['fraud_approved']:.0f}"
)

print(
    f"Fraud review          : "
    f"{best_policy['fraud_review']:.0f}"
)

print(
    f"Fraud blocked         : "
    f"{best_policy['fraud_blocked']:.0f}"
)

print(
    f"Legitimate blocked    : "
    f"{best_policy['legitimate_blocked']:.0f}"
)

print(
    f"Fraud capture rate    : "
    f"{best_policy['fraud_capture_rate']:.4%}"
)

TWO-THRESHOLD POLICY OPTIMIZATION
Best medium threshold : 0.01
Best high threshold   : 0.18
Prototype cost        : 162

Fraud approved        : 16
Fraud review          : 5
Fraud blocked         : 74
Legitimate blocked    : 2
Fraud capture rate    : 83.1579%


In [12]:
print("Keys in train_test_split.pkl:")
print(split_data.keys())

Keys in train_test_split.pkl:
dict_keys(['X_train', 'X_test', 'y_train', 'y_test'])


In [13]:
from sklearn.model_selection import train_test_split

# =========================================================
# CREATE VALIDATION SET FROM TRAINING DATA
# =========================================================

X_train = split_data["X_train"]
y_train = split_data["y_train"]

X_test = split_data["X_test"]
y_test = split_data["y_test"]


X_train_new, X_val, y_train_new, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)


print("=" * 60)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)

print("\nTraining set:")
print("X_train:", X_train_new.shape)
print("Fraud:", int(y_train_new.sum()))

print("\nValidation set:")
print("X_val:", X_val.shape)
print("Fraud:", int(y_val.sum()))

print("\nFinal test set:")
print("X_test:", X_test.shape)
print("Fraud:", int(y_test.sum()))

TRAIN / VALIDATION / TEST SPLIT

Training set:
X_train: (181584, 30)
Fraud: 302

Validation set:
X_val: (45396, 30)
Fraud: 76

Final test set:
X_test: (56746, 30)
Fraud: 95


In [14]:
# =========================================================
# GENERATE VALIDATION FRAUD PROBABILITIES
# =========================================================

X_val_model = X_val[
    MODEL_FEATURES
]

X_val_scaled = scaler.transform(
    X_val_model
)

y_val_probability = model.predict_proba(
    X_val_scaled
)[:, 1]


print("=" * 60)
print("VALIDATION PREDICTIONS")
print("=" * 60)

print("Validation samples:", len(y_val))

print(
    "Validation fraud cases:",
    int(y_val.sum())
)

print(
    "Minimum probability:",
    y_val_probability.min()
)

print(
    "Maximum probability:",
    y_val_probability.max()
)

VALIDATION PREDICTIONS
Validation samples: 45396
Validation fraud cases: 76
Minimum probability: 1.7975435e-07
Maximum probability: 0.99918777


In [15]:
# =========================================================
# TWO-THRESHOLD OPTIMIZATION ON VALIDATION SET
# =========================================================

FN_COST = 10
FP_COST = 1

threshold_values = np.arange(
    0.01,
    0.51,
    0.01
)

validation_results = []


for medium_threshold in threshold_values:

    for high_threshold in threshold_values:

        if high_threshold <= medium_threshold:
            continue

        # -------------------------------------------------
        # Assign risk decisions
        # -------------------------------------------------

        risk = np.where(
            y_val_probability < medium_threshold,
            "APPROVE",
            np.where(
                y_val_probability < high_threshold,
                "REVIEW",
                "BLOCK"
            )
        )

        # -------------------------------------------------
        # Actual fraud outcomes
        # -------------------------------------------------

        fraud_approved = np.sum(
            (y_val.values == 1) &
            (risk == "APPROVE")
        )

        fraud_review = np.sum(
            (y_val.values == 1) &
            (risk == "REVIEW")
        )

        fraud_blocked = np.sum(
            (y_val.values == 1) &
            (risk == "BLOCK")
        )

        # -------------------------------------------------
        # Legitimate transactions blocked
        # -------------------------------------------------

        legitimate_blocked = np.sum(
            (y_val.values == 0) &
            (risk == "BLOCK")
        )

        # -------------------------------------------------
        # Cost
        # -------------------------------------------------

        total_cost = (
            fraud_approved * FN_COST
            +
            legitimate_blocked * FP_COST
        )

        # -------------------------------------------------
        # Fraud captured by REVIEW + BLOCK
        # -------------------------------------------------

        fraud_capture_rate = (
            fraud_review + fraud_blocked
        ) / y_val.sum()

        validation_results.append({

            "medium_threshold":
                medium_threshold,

            "high_threshold":
                high_threshold,

            "cost":
                total_cost,

            "fraud_approved":
                fraud_approved,

            "fraud_review":
                fraud_review,

            "fraud_blocked":
                fraud_blocked,

            "legitimate_blocked":
                legitimate_blocked,

            "fraud_capture_rate":
                fraud_capture_rate
        })


validation_policy_df = pd.DataFrame(
    validation_results
)


# =========================================================
# BEST POLICY
# =========================================================

best_validation_policy = (
    validation_policy_df
    .sort_values(
        [
            "cost",
            "fraud_capture_rate"
        ],
        ascending=[
            True,
            False
        ]
    )
    .iloc[0]
)


print("=" * 60)
print("VALIDATION THRESHOLD OPTIMIZATION")
print("=" * 60)

print(
    f"Best medium threshold : "
    f"{best_validation_policy['medium_threshold']:.2f}"
)

print(
    f"Best high threshold   : "
    f"{best_validation_policy['high_threshold']:.2f}"
)

print(
    f"Validation cost       : "
    f"{best_validation_policy['cost']:.0f}"
)

print()

print(
    f"Fraud approved        : "
    f"{best_validation_policy['fraud_approved']:.0f}"
)

print(
    f"Fraud review          : "
    f"{best_validation_policy['fraud_review']:.0f}"
)

print(
    f"Fraud blocked         : "
    f"{best_validation_policy['fraud_blocked']:.0f}"
)

print(
    f"Legitimate blocked    : "
    f"{best_validation_policy['legitimate_blocked']:.0f}"
)

print(
    f"Fraud capture rate    : "
    f"{best_validation_policy['fraud_capture_rate']:.4%}"
)

VALIDATION THRESHOLD OPTIMIZATION
Best medium threshold : 0.01
Best high threshold   : 0.19
Validation cost       : 91

Fraud approved        : 9
Fraud review          : 2
Fraud blocked         : 65
Legitimate blocked    : 1
Fraud capture rate    : 88.1579%


In [16]:
# =========================================================
# FINAL TEST EVALUATION
# USING VALIDATION-SELECTED THRESHOLDS
# =========================================================

MEDIUM_THRESHOLD = float(
    best_validation_policy["medium_threshold"]
)

HIGH_THRESHOLD = float(
    best_validation_policy["high_threshold"]
)

FN_COST = 10
FP_COST = 1


# =========================================================
# APPLY LOCKED POLICY TO FINAL TEST SET
# =========================================================

test_risk = np.where(
    y_probability < MEDIUM_THRESHOLD,
    "APPROVE",
    np.where(
        y_probability < HIGH_THRESHOLD,
        "REVIEW",
        "BLOCK"
    )
)


# =========================================================
# CREATE FINAL TEST POLICY DATAFRAME
# =========================================================

final_test_policy = pd.DataFrame({

    "actual": y_test.values,

    "probability": y_probability,

    "risk_level": np.where(
        test_risk == "APPROVE",
        "LOW",
        np.where(
            test_risk == "REVIEW",
            "MEDIUM",
            "HIGH"
        )
    ),

    "action": test_risk
})


# =========================================================
# TRANSACTION ROUTING
# =========================================================

routing = (
    final_test_policy["action"]
    .value_counts()
    .reindex(
        ["APPROVE", "REVIEW", "BLOCK"],
        fill_value=0
    )
)


# =========================================================
# ACTUAL FRAUD BY ACTION
# =========================================================

fraud_by_action = (
    final_test_policy[
        final_test_policy["actual"] == 1
    ]["action"]
    .value_counts()
    .reindex(
        ["APPROVE", "REVIEW", "BLOCK"],
        fill_value=0
    )
)


# =========================================================
# LEGITIMATE BY ACTION
# =========================================================

legitimate_by_action = (
    final_test_policy[
        final_test_policy["actual"] == 0
    ]["action"]
    .value_counts()
    .reindex(
        ["APPROVE", "REVIEW", "BLOCK"],
        fill_value=0
    )
)


# =========================================================
# COST COMPONENTS
# =========================================================

fraud_approved = int(
    (
        (final_test_policy["actual"] == 1) &
        (final_test_policy["action"] == "APPROVE")
    ).sum()
)

fraud_review = int(
    (
        (final_test_policy["actual"] == 1) &
        (final_test_policy["action"] == "REVIEW")
    ).sum()
)

fraud_blocked = int(
    (
        (final_test_policy["actual"] == 1) &
        (final_test_policy["action"] == "BLOCK")
    ).sum()
)

legitimate_blocked = int(
    (
        (final_test_policy["actual"] == 0) &
        (final_test_policy["action"] == "BLOCK")
    ).sum()
)


# =========================================================
# FINAL COST
# =========================================================

total_cost = (
    fraud_approved * FN_COST
    +
    legitimate_blocked * FP_COST
)


# =========================================================
# FRAUD CAPTURE
# =========================================================

fraud_capture_rate = (
    fraud_review + fraud_blocked
) / int(y_test.sum())


# =========================================================
# FINAL RESULTS
# =========================================================

print("=" * 60)
print("FINAL TEST SET - LOCKED RISK POLICY")
print("=" * 60)

print(
    f"Medium threshold : {MEDIUM_THRESHOLD:.2%}"
)

print(
    f"High threshold   : {HIGH_THRESHOLD:.2%}"
)

print()

print("Transaction Routing")
print("--------------------")

print(routing)

print()

print("Actual Fraud by Action")
print("-----------------------")

print(fraud_by_action)

print()

print("Legitimate by Action")
print("---------------------")

print(legitimate_by_action)

print()

print("Final Policy Outcomes")
print("----------------------")

print(
    f"Fraud approved     : {fraud_approved}"
)

print(
    f"Fraud review       : {fraud_review}"
)

print(
    f"Fraud blocked      : {fraud_blocked}"
)

print(
    f"Legitimate blocked : {legitimate_blocked}"
)

print()

print(
    f"Fraud capture rate : {fraud_capture_rate:.4%}"
)

print(
    f"False-negative cost: "
    f"{fraud_approved} × {FN_COST}"
)

print(
    f"False-positive cost: "
    f"{legitimate_blocked} × {FP_COST}"
)

print(
    f"Total prototype cost: {total_cost}"
)

FINAL TEST SET - LOCKED RISK POLICY
Medium threshold : 1.00%
High threshold   : 19.00%

Transaction Routing
--------------------
action
APPROVE    56604
REVIEW        67
BLOCK         75
Name: count, dtype: int64

Actual Fraud by Action
-----------------------
action
APPROVE    16
REVIEW      6
BLOCK      73
Name: count, dtype: int64

Legitimate by Action
---------------------
action
APPROVE    56588
REVIEW        61
BLOCK          2
Name: count, dtype: int64

Final Policy Outcomes
----------------------
Fraud approved     : 16
Fraud review       : 6
Fraud blocked      : 73
Legitimate blocked : 2

Fraud capture rate : 83.1579%
False-negative cost: 16 × 10
False-positive cost: 2 × 1
Total prototype cost: 162


In [17]:
# =========================================================
# FINAL TEST METRICS FOR LOCKED POLICY
# =========================================================

# Treat REVIEW + BLOCK as "detected / flagged"
# APPROVE = not detected

y_policy_pred = (
    final_test_policy["action"] != "APPROVE"
).astype(int)


# ---------------------------------------------------------
# Confusion Matrix
# ---------------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    final_test_policy["actual"],
    y_policy_pred
).ravel()


# ---------------------------------------------------------
# Metrics
# ---------------------------------------------------------

policy_precision = precision_score(
    final_test_policy["actual"],
    y_policy_pred,
    zero_division=0
)

policy_recall = recall_score(
    final_test_policy["actual"],
    y_policy_pred,
    zero_division=0
)

policy_f1 = f1_score(
    final_test_policy["actual"],
    y_policy_pred,
    zero_division=0
)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("=" * 60)
print("FINAL LOCKED POLICY METRICS")
print("=" * 60)

print(
    f"Precision : {policy_precision:.4f}"
)

print(
    f"Recall    : {policy_recall:.4f}"
)

print(
    f"F1 Score  : {policy_f1:.4f}"
)

print()

print("Confusion Matrix")
print("----------------")

print(f"TN : {tn}")
print(f"FP : {fp}")
print(f"FN : {fn}")
print(f"TP : {tp}")

FINAL LOCKED POLICY METRICS
Precision : 0.5563
Recall    : 0.8316
F1 Score  : 0.6667

Confusion Matrix
----------------
TN : 56588
FP : 63
FN : 16
TP : 79


In [18]:
# =========================================================
# FINAL ERROR ANALYSIS
# =========================================================

# Add prediction information to the final test dataframe

error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["fraud_probability"] = y_probability
error_df["action"] = final_test_policy["action"].values


# =========================================================
# 1. FALSE NEGATIVES
# =========================================================
# Actual fraud but system approved

false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["action"] == "APPROVE")
].copy()


# =========================================================
# 2. FALSE POSITIVES
# =========================================================
# Legitimate but system sent to REVIEW/BLOCK

false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["action"] != "APPROVE")
].copy()


print("=" * 60)
print("FINAL ERROR ANALYSIS")
print("=" * 60)

print("\nFalse Negatives")
print("----------------")
print("Missed fraud transactions:", len(false_negatives))

print(
    "\nFraud probability statistics:"
)

print(
    false_negatives["fraud_probability"].describe()
)


print("\nFalse Positives")
print("----------------")
print(
    "Legitimate transactions flagged:",
    len(false_positives)
)

print(
    "\nFraud probability statistics:"
)

print(
    false_positives["fraud_probability"].describe()
)

FINAL ERROR ANALYSIS

False Negatives
----------------
Missed fraud transactions: 16

Fraud probability statistics:
count    16.000000
mean      0.001382
std       0.002744
min       0.000006
25%       0.000036
50%       0.000064
75%       0.000620
max       0.007974
Name: fraud_probability, dtype: float64

False Positives
----------------
Legitimate transactions flagged: 63

Fraud probability statistics:
count    63.000000
mean      0.059806
std       0.142162
min       0.010401
25%       0.012700
50%       0.020695
75%       0.048285
max       0.989354
Name: fraud_probability, dtype: float64


In [19]:
# =========================================================
# FALSE POSITIVE BREAKDOWN
# =========================================================

fp_review = false_positives[
    false_positives["action"] == "REVIEW"
]

fp_block = false_positives[
    false_positives["action"] == "BLOCK"
]


print("=" * 60)
print("FALSE POSITIVE BREAKDOWN")
print("=" * 60)

print(
    "\nLegitimate transactions sent to REVIEW:",
    len(fp_review)
)

print(
    "Legitimate transactions BLOCKED:",
    len(fp_block)
)


print("\nREVIEW probability statistics")
print("--------------------------------")

print(
    fp_review["fraud_probability"].describe()
)


print("\nBLOCK probability statistics")
print("--------------------------------")

print(
    fp_block["fraud_probability"].describe()
)

FALSE POSITIVE BREAKDOWN

Legitimate transactions sent to REVIEW: 61
Legitimate transactions BLOCKED: 2

REVIEW probability statistics
--------------------------------
count    61.000000
mean      0.036004
std       0.036864
min       0.010401
25%       0.012672
50%       0.019937
75%       0.037649
max       0.176655
Name: fraud_probability, dtype: float64

BLOCK probability statistics
--------------------------------
count    2.000000
mean     0.785763
std      0.287922
min      0.582172
25%      0.683967
50%      0.785763
75%      0.887559
max      0.989354
Name: fraud_probability, dtype: float64


In [20]:
# =========================================================
# SELECT REPRESENTATIVE ERROR CASES
# =========================================================

# One missed fraud
missed_fraud = (
    false_negatives
    .sort_values("fraud_probability")
    .iloc[0]
)

# One legitimate review case
legitimate_review = (
    fp_review
    .sort_values(
        "fraud_probability",
        ascending=False
    )
    .iloc[0]
)

# One legitimate blocked case
legitimate_block = (
    fp_block
    .sort_values(
        "fraud_probability",
        ascending=False
    )
    .iloc[0]
)


print("=" * 60)
print("REPRESENTATIVE ERROR CASES")
print("=" * 60)

print("\n1. MISSED FRAUD")
print("----------------")
print(
    "Probability:",
    missed_fraud["fraud_probability"]
)

print(
    "Amount:",
    missed_fraud["Amount"]
)

print(
    "Time:",
    missed_fraud["Time"]
)


print("\n2. LEGITIMATE → REVIEW")
print("-----------------------")
print(
    "Probability:",
    legitimate_review["fraud_probability"]
)

print(
    "Amount:",
    legitimate_review["Amount"]
)

print(
    "Time:",
    legitimate_review["Time"]
)


print("\n3. LEGITIMATE → BLOCK")
print("----------------------")
print(
    "Probability:",
    legitimate_block["fraud_probability"]
)

print(
    "Amount:",
    legitimate_block["Amount"]
)

print(
    "Time:",
    legitimate_block["Time"]
)

REPRESENTATIVE ERROR CASES

1. MISSED FRAUD
----------------
Probability: 6.2543068e-06
Amount: 1.18
Time: 53076.0

2. LEGITIMATE → REVIEW
-----------------------
Probability: 0.17665464
Amount: 107.5
Time: 127631.0

3. LEGITIMATE → BLOCK
----------------------
Probability: 0.9893544
Amount: 109.9
Time: 75706.0


In [21]:
import shap
import pandas as pd


# =========================================================
# SHAP EXPLAINER
# =========================================================

explainer = shap.TreeExplainer(model)


# =========================================================
# FUNCTION TO EXPLAIN ONE TRANSACTION
# =========================================================

def explain_transaction(row):

    transaction = row[
        MODEL_FEATURES
    ].to_frame().T

    transaction_scaled = scaler.transform(
        transaction
    )

    shap_explanation = explainer(
        transaction_scaled
    )

    shap_values = shap_explanation.values[0]

    explanation_df = pd.DataFrame({

        "feature": MODEL_FEATURES,

        "shap_value": shap_values,

        "absolute_shap": np.abs(
            shap_values
        )

    })

    explanation_df = (
        explanation_df
        .sort_values(
            "absolute_shap",
            ascending=False
        )
        .head(10)
    )

    return explanation_df[
        [
            "feature",
            "shap_value"
        ]
    ]


# =========================================================
# 1. MISSED FRAUD
# =========================================================

missed_fraud_shap = explain_transaction(
    missed_fraud
)


print("=" * 60)
print("SHAP - MISSED FRAUD")
print("=" * 60)

print(
    missed_fraud_shap.to_string(
        index=False
    )
)


# =========================================================
# 2. LEGITIMATE → REVIEW
# =========================================================

legitimate_review_shap = explain_transaction(
    legitimate_review
)


print("\n" + "=" * 60)
print("SHAP - LEGITIMATE → REVIEW")
print("=" * 60)

print(
    legitimate_review_shap.to_string(
        index=False
    )
)


# =========================================================
# 3. LEGITIMATE → BLOCK
# =========================================================

legitimate_block_shap = explain_transaction(
    legitimate_block
)


print("\n" + "=" * 60)
print("SHAP - LEGITIMATE → BLOCK")
print("=" * 60)

print(
    legitimate_block_shap.to_string(
        index=False
    )
)

SHAP - MISSED FRAUD
feature  shap_value
    V12   -0.993150
     V4   -0.835831
    V15   -0.805385
    V14   -0.757732
    V10   -0.726195
    V11   -0.637034
     V3   -0.444629
     V6    0.370284
    V25   -0.364608
    V27    0.280070

SHAP - LEGITIMATE → REVIEW
feature  shap_value
    V14    4.167737
    V10    2.496476
     V4   -0.846864
    V16   -0.603965
     V6    0.436249
     V3    0.368927
    V17   -0.338854
 Amount    0.313896
   Time   -0.272748
    V28    0.243493

SHAP - LEGITIMATE → BLOCK
feature  shap_value
    V14    4.105653
    V10    2.250525
    V12    1.286989
 Amount    1.273797
     V4    0.930305
    V28   -0.614346
    V16    0.605146
    V17    0.518917
     V7    0.493916
    V26   -0.445302


In [23]:
from pathlib import Path
import joblib

# Project root
BASE_DIR = Path.cwd().parent

# Models directory
MODELS_DIR = BASE_DIR / "models"

print("Models directory:", MODELS_DIR)

Models directory: C:\Users\ANKAN SEN\Desktop\AI-Risk-Manager\models


In [24]:
# =========================================================
# SAVE FINAL VALIDATED RISK POLICY
# =========================================================

final_risk_policy = {
    "model": "XGBoost",
    "threshold_medium": 0.01,
    "threshold_high": 0.19,
    "fn_cost": 10,
    "fp_cost": 1,
    "threshold_selection": "validation_set",
    "validation_size": len(y_val),
    "final_test_size": len(y_test)
}

policy_path = MODELS_DIR / "risk_policy.pkl"

joblib.dump(
    final_risk_policy,
    policy_path
)

print("=" * 60)
print("FINAL RISK POLICY SAVED")
print("=" * 60)

print("Path:", policy_path)

print(
    "Medium threshold:",
    final_risk_policy["threshold_medium"]
)

print(
    "High threshold:",
    final_risk_policy["threshold_high"]
)

print("FN cost:", final_risk_policy["fn_cost"])
print("FP cost:", final_risk_policy["fp_cost"])

FINAL RISK POLICY SAVED
Path: C:\Users\ANKAN SEN\Desktop\AI-Risk-Manager\models\risk_policy.pkl
Medium threshold: 0.01
High threshold: 0.19
FN cost: 10
FP cost: 1
